In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abozeidmohamed/dataset/label_encoder_classes (1).npy
/kaggle/input/datasets/abozeidmohamed/dataset/encoded_labels_under (2).npy
/kaggle/input/datasets/abozeidmohamed/dataset/images_under (1).npy
/kaggle/input/datasets/abozeidmohamed/dataset1/label_encoder_classes (1).npy
/kaggle/input/datasets/abozeidmohamed/dataset1/images_over.npy
/kaggle/input/datasets/abozeidmohamed/dataset1/encoded_labels_over (2).npy


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

# --- تحميل البيانات --- ✅ الترتيب الصح
X = np.load('/kaggle/input/datasets/abozeidmohamed/dataset/images_under (1).npy')        # ← الصور
y = np.load('/kaggle/input/datasets/abozeidmohamed/dataset/encoded_labels_under (2).npy')  # ← اللابيلز

# لو الصور رمادية نكررها لثلاث قنوات
if X.shape[-1] == 1:
    X = np.repeat(X, 3, axis=-1)

# ─── التقسيم الثلاثي 70 / 15 / 15 ───────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)

# ─── التحقق ──────────────────────────────────────────────────
total = len(X)
print(f"Total  samples : {total}")
print(f"Train  samples : {len(X_train)}  ({len(X_train)/total*100:.1f}%)")
print(f"Val    samples : {len(X_val)}   ({len(X_val)/total*100:.1f}%)")
print(f"Test   samples : {len(X_test)}  ({len(X_test)/total*100:.1f}%)")

input_shape = X_train.shape[1:]
num_classes = len(np.unique(y))

2026-05-21 14:59:51.636825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779375591.855607      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779375591.926155      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779375592.411520      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779375592.411568      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779375592.411573      57 computation_placer.cc:177] computation placer alr

Total  samples : 32128
Train  samples : 22488  (70.0%)
Val    samples : 4820   (15.0%)
Test   samples : 4820  (15.0%)


In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)


In [4]:
def build_mobilenetv2(input_shape, num_classes):
    base_model = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )

    # Fine-tuning: نفتح آخر 20 طبقة
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    for layer in base_model.layers[-20:]:
        layer.trainable = True

    inputs = Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [7]:
import gc, time, os, psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K

from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ═════════════════════════════════════
# SAFE FLOPs & MACs (FIXED)
# ═════════════════════════════════════
def flops_macs_keras(model):
    total_macs = 0

    for layer in model.layers:

        # Conv2D
        if isinstance(layer, tf.keras.layers.Conv2D):
            weights = layer.get_weights()
            if len(weights) > 0:
                kH, kW, in_ch, out_ch = weights[0].shape
                if layer.output_shape is not None and len(layer.output_shape) == 4:
                    _, H, W, _ = layer.output_shape
                    total_macs += H * W * out_ch * kH * kW * in_ch

        # DepthwiseConv2D
        elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
            weights = layer.get_weights()
            if len(weights) > 0:
                kH, kW, in_ch, depth_mult = weights[0].shape
                if layer.output_shape is not None and len(layer.output_shape) == 4:
                    _, H, W, C = layer.output_shape
                    total_macs += H * W * C * kH * kW

        # Dense (FIXED)
        elif isinstance(layer, tf.keras.layers.Dense):
            weights = layer.get_weights()
            if len(weights) > 0:
                kernel = weights[0]  # (in_features, out_features)
                total_macs += kernel.shape[0] * kernel.shape[1]

    total_flops = total_macs * 2
    return total_flops, total_macs


# ═════════════════════════════════════
# SETTINGS
# ═════════════════════════════════════
SAVE_DIR = "/kaggle/working/"
os.makedirs(SAVE_DIR, exist_ok=True)

num_runs = 10
results = []
all_test_probs = []

# ═════════════════════════════════════
# BUILD MODEL ONCE FOR FLOPs
# ═════════════════════════════════════
print("🔢 Calculating FLOPs & MACs...")

_temp_model = build_mobilenetv2(input_shape, num_classes)
model_flops, model_macs = flops_macs_keras(_temp_model)

K.clear_session()
del _temp_model
gc.collect()

print(f"  MACs  : {model_macs/1e6:.2f} M")
print(f"  FLOPs : {model_flops/1e6:.2f} M")


# ═════════════════════════════════════
# MAIN LOOP
# ═════════════════════════════════════
for run in range(1, num_runs + 1):

    print(f"\n🚀 Run {run}/{num_runs}")

    start_time = time.time()
    start_mem = psutil.Process().memory_info().rss / (1024 ** 2)

    model = build_mobilenetv2(input_shape, num_classes)

    early_stop = EarlyStopping(patience=7, restore_best_weights=True, monitor='val_loss')
    lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=64),
        validation_data=(X_val, y_val),
        epochs=20,
        callbacks=[early_stop, lr_reduce],
        verbose=1
    )

    train_time = time.time() - start_time
    train_mem = psutil.Process().memory_info().rss / (1024 ** 2) - start_mem

    # ══ inference ══
    _ = model.predict(X_val[:8], verbose=0)
    inf_start = time.time()
    y_pred_prob = model.predict(X_val, verbose=0)
    inf_time = time.time() - inf_start

    y_pred = np.argmax(y_pred_prob, axis=1)

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

    cm = confusion_matrix(y_val, y_pred)
    mcc = matthews_corrcoef(y_val, y_pred)

    y_val_bin = label_binarize(y_val, classes=np.arange(num_classes))
    try:
        roc_auc = roc_auc_score(y_val_bin, y_pred_prob, multi_class='ovr')
    except:
        roc_auc = np.nan

    f1 = f1_score(y_val, y_pred, average='weighted')
    prec = precision_score(y_val, y_pred, average='weighted')
    rec = recall_score(y_val, y_pred, average='weighted')

    print(f"Acc={val_acc:.4f} | MCC={mcc:.4f} | F1={f1:.4f}")

    # ══ test probs ══
    _ = model.predict(X_test[:8], verbose=0)
    test_probs = model.predict(X_test, verbose=0)
    all_test_probs.append(test_probs)

    np.save(os.path.join(SAVE_DIR, f"test_probs_run{run}.npy"), test_probs)
    model.save(os.path.join(SAVE_DIR, f"model_run{run}.h5"))

    results.append({
        "Run": run,
        "Val_Accuracy": val_acc,
        "Val_Loss": val_loss,
        "MCC": mcc,
        "ROC_AUC": roc_auc,
        "F1": f1,
        "Precision": prec,
        "Recall": rec,
        "Train_Time": train_time,
        "Inference_Time": inf_time,
        "Memory_MB": train_mem,
        "MACs(M)": model_macs / 1e6,
        "FLOPs(M)": model_flops / 1e6
    })

    del model, history, y_pred, y_pred_prob, test_probs
    K.clear_session()
    gc.collect()


# ═════════════════════════════════════
# ENSEMBLE AVERAGE
# ═════════════════════════════════════
mob_test_probs = np.mean(np.stack(all_test_probs, axis=0), axis=0)
np.save(os.path.join(SAVE_DIR, "mob_test_probs.npy"), mob_test_probs)

print("\n✔ Ensemble saved:", mob_test_probs.shape)


# ═════════════════════════════════════
# FINAL DF
# ═════════════════════════════════════
df = pd.DataFrame(results)

print("\n========== AVERAGE RESULTS ==========")
print(df.mean(numeric_only=True))

🔢 Calculating FLOPs & MACs...


/tmp/ipykernel_57/3345950361.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


  MACs  : 0.34 M
  FLOPs : 0.67 M

🚀 Run 1/10
Epoch 1/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 57s 120ms/step - accuracy: 0.1521 - loss: 3.7108 - val_accuracy: 0.4164 - val_loss: 2.0921 - learning_rate: 1.0000e-04
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 28s 81ms/step - accuracy: 0.5007 - loss: 1.6482 - val_accuracy: 0.5591 - val_loss: 1.5795 - learning_rate: 1.0000e-04
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 28s 81ms/step - accuracy: 0.6434 - loss: 1.1391 - val_accuracy: 0.6755 - val_loss: 1.1215 - learning_rate: 1.0000e-04
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 28s 80ms/step - accuracy: 0.7179 - loss: 0.8782 - val_accuracy: 0.7548 - val_loss: 0.8181 - learning_rate: 1.0000e-04
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 28s 80ms/step - accuracy: 0.7835 - loss: 0.6704 - val_accuracy: 0.8178 - val_loss: 0.5976 - learning_rate: 1.0000e-04
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 28s 80ms/step - accuracy: 0.8187 - loss: 0.5692 - val_accuracy: 0.8467 - val_loss: 0.5166 - learning_rate: 1.0000e-04
Epoch


🚀 Run 2/10


/tmp/ipykernel_57/3345950361.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


...

/tmp/ipykernel_57/3345950361.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


352/352 ━━━━━━━━━━━━━━━━━━━━ 54s 115ms/step - accuracy: 0.1506 - loss: 3.6869 - val_accuracy: 0.3803 - val_loss: 2.1377 - learning_rate: 1.0000e-04
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 76ms/step - accuracy: 0.4954 - loss: 1.7051 - val_accuracy: 0.5774 - val_loss: 1.3869 - learning_rate: 1.0000e-04
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 76ms/step - accuracy: 0.6347 - loss: 1.1615 - val_accuracy: 0.6766 - val_loss: 1.1201 - learning_rate: 1.0000e-04
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 76ms/step - accuracy: 0.7185 - loss: 0.8786 - val_accuracy: 0.7834 - val_loss: 0.6988 - learning_rate: 1.0000e-04
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 77ms/step - accuracy: 0.7774 - loss: 0.6969 - val_accuracy: 0.8320 - val_loss: 0.5153 - learning_rate: 1.0000e-04
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 77ms/step - accuracy: 0.8141 - loss: 0.5781 - val_accuracy: 0.8645 - val_loss: 0.4142 - learning_rate: 1.0000e-04
Epoch 7/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 77ms/step - accura


✔ Ensemble saved: (4820, 32)

========== AVERAGE RESULTS ==========
Run                 5.500000
Val_Accuracy        0.970934
Val_Loss            0.092831
MCC                 0.970016
ROC_AUC             0.999722
F1                  0.970933
Precision           0.971538
Recall              0.970934
Train_Time        584.589192
Inference_Time      8.007532
Memory_MB         416.499609
MACs(M)             0.335872
FLOPs(M)            0.671744
dtype: float64


In [8]:
import tensorflow as tf
import pandas as pd
import os

# ── اختار أحسن run بناءً على Val Accuracy ────────────────────
df_metrics = pd.DataFrame(results)
best_run    = int(df_metrics.loc[df_metrics['Val_Accuracy'].idxmax(), 'Run'])
print(f"  🏆 Best Run: {best_run}  (Val Acc: {df_metrics['Val_Accuracy'].max():.4f})")

# ── تحميل الموديل ─────────────────────────────────────────────
model_path = f"/kaggle/working/model_run{best_run}.h5"
model      = tf.keras.models.load_model(model_path)
print(f"  ✅ Model loaded from: {model_path}")
print(f"  📐 Input shape: {model.input_shape}")

  🏆 Best Run: 10  (Val Acc: 0.9755)


  ✅ Model loaded from: /kaggle/working/model_run10.h5
  📐 Input shape: (None, 64, 64, 3)


In [9]:
import numpy as np

# ── تحميل الـ class names ─────────────────────────────────────
class_names = np.load(
    '/kaggle/input/datasets/abozeidmohamed/dataset1/label_encoder_classes (1).npy',
    allow_pickle=True
)

print(f"✅ class_names loaded: {class_names}")
print(f"   عدد الكلاسات: {len(class_names)}")

✅ class_names loaded: ['ain' 'al' 'aleff' 'bb' 'dal' 'dha' 'dhad' 'fa' 'gaaf' 'ghain' 'ha' 'haa'
 'jeem' 'kaaf' 'khaa' 'la' 'laam' 'meem' 'nun' 'ra' 'saad' 'seen' 'sheen'
 'ta' 'taa' 'thaa' 'thal' 'toot' 'waw' 'ya' 'yaa' 'zay']
   عدد الكلاسات: 32


In [10]:
import tensorflow as tf
from tensorflow.python.framework import convert_to_constants

# ─────────────────────────────────────────────
# 1) نجيب input shape من الموديل
# ─────────────────────────────────────────────
input_shape = model.input_shape

# لو فيه batch None نخليه 1
input_shape = tuple([1 if x is None else x for x in input_shape])

print("  📐 Used input shape for FLOPs:", input_shape)


# ─────────────────────────────────────────────
# 2) FLOPs function
# ─────────────────────────────────────────────
def get_flops(model, input_shape):

    full_model = tf.function(lambda x: model(x))
    full_model = full_model.get_concrete_function(
        tf.TensorSpec(input_shape, tf.float32)
    )

    frozen_func = convert_to_constants.convert_variables_to_constants_v2(full_model)
    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.import_graph_def(graph_def, name="")

        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd="op",
            options=opts
        )

        return flops.total_float_ops


# ─────────────────────────────────────────────
# 3) الحساب
# ─────────────────────────────────────────────
flops = get_flops(model, input_shape)

flops_m = flops / 1e6
macs_m  = flops_m / 2   # approximation

# ─────────────────────────────────────────────
# 4) output
# ─────────────────────────────────────────────
print("\n==============================")
print("  ⚡ Model Complexity")
print("==============================")
print(f"FLOPs  : {flops_m:.4f} M")
print(f"MACs   : {macs_m:.4f} M")

  📐 Used input shape for FLOPs: (1, 64, 64, 3)


I0000 00:00:1779383698.493780      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1779383698.494028      57 single_machine.cc:374] Starting new session
I0000 00:00:1779383698.495115      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.



=========================Options=============================
-max_depth                  10000
-min_bytes                  0
-min_peak_bytes             0
-min_residual_bytes         0
-min_output_bytes           0
-min_micros                 0
-min_accelerator_micros     0
-min_cpu_micros             0
-min_params                 0
-min_float_ops              1
-min_occurrence             0
-step                       -1
-order_by                   float_ops
-account_type_regexes       .*
-start_name_regexes         .*
-trim_name_regexes          
-show_name_regexes          .*
-hide_name_regexes          
-account_displayed_op_only  true
-select                     float_ops
-output                     stdout:

==================Model Analysis Report======================

Doc:
op: The nodes are operation kernel type, such as MatMul, Conv2D. Graph nodes belonging to the same type are aggregated together.
flops: Number of float operations. Note: Please read the implementation for th